# Caracal s07 - Bench suite Kaggle T4 x2

Roda TODOS os benches (13) contra o ultimo checkpoint publicado:
`arturpn/caracal-base-3b-s04` (Kaggle dataset publico 07-03, 239MB, LoRA adapter Qwen2.5-3B).

Output publicado como `vitorscrt/caracal-bench-s07-v0`.

**Settings:**
1. Accelerator -> GPU T4 x2
2. Internet ON
3. Persistence Variables and Files

**Save & Run All** (nao mexe em nada, ja ta pronto).

In [ ]:
CHECKPOINT_DATASET = 'arturpn/caracal-base-3b-s04'
BASE_MODEL = 'unsloth/Qwen2.5-Coder-3B-Instruct'
OUTPUT_DATASET = 'vitorscrt/caracal-bench-s07-v0'
print(f'checkpoint={CHECKPOINT_DATASET} -> bench s07 completo -> {OUTPUT_DATASET}')

In [ ]:
!pip install -q -U 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=1.0.0' 'sentence-transformers>=3.0.0' 'scipy>=1.13.0' 'torchao>=0.16.0' kaggle

In [ ]:
import os, subprocess

if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
rev = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'cloned, HEAD={rev}')

In [ ]:
import torch
print(f'CUDA={torch.cuda.is_available()} n_gpu={torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  gpu{i}: {torch.cuda.get_device_name(i)}')

In [ ]:
import subprocess
ckpt_dir = '/kaggle/working/ckpt'
subprocess.run(['kaggle', 'datasets', 'download', '-d', CHECKPOINT_DATASET,
                '-p', ckpt_dir, '--unzip', '--force'], check=True)
subprocess.run(['ls', '-la', ckpt_dir], check=True)

In [ ]:
import subprocess
os.makedirs('data', exist_ok=True)
subprocess.run(['wget', '-q', '-O', 'data/cwec_latest.xml.zip',
                'https://cwe.mitre.org/data/xml/cwec_latest.xml.zip'], check=True)
subprocess.run(['unzip', '-oq', 'data/cwec_latest.xml.zip', '-d', 'data/'], check=True)
print('CWE XML pronto')

In [ ]:
import subprocess, sys
os.makedirs('/kaggle/working/bench-out', exist_ok=True)
BENCHES = ['cti_bench', 'cybermetric', 'secqa', 'secbench', 'mmlu_security',
           'cwe_prediction', 'seceval', 'cybersoceval', 'primevul']
# skip: cs_eval (HF slug fantasma), cybercert (sem HF public),
#       cybench_kaggle + nyu_ctf_kaggle (datasets iterate-labs-ai ainda nao subidos)
cmd = [sys.executable, '-u', '-m', 'eval.s07.run_all_benches',
       '--model', BASE_MODEL,
       '--adapter', ckpt_dir,
       '--benches', *BENCHES,
       '--n-cti-rcm', '1000', '--n-cti-mcq', '500',
       '--cybermetric-tier', '500',
       '--n-secbench', '1000', '--n-mmlu', '100',
       '--n-cwe-pred', '500', '--n-seceval', '500',
       '--n-cybersoceval', '500', '--n-primevul', '500',
       '--out', '/kaggle/working/bench-out/s07-eval.json']
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import json
with open('/kaggle/working/bench-out/s07-eval.json') as f:
    results = json.load(f)
print(f'MODEL base={results.get("model")} adapter={results.get("adapter")}')
for bench, res in results.items():
    if bench in ('model', 'adapter', 'mcnemar_vs_prev') or not isinstance(res, dict):
        continue
    if 'accuracy' in res:
        print(f"{bench:20s}: {res['accuracy']*100:5.1f}%  n={res['n']:4d}  "
              f"CI[{res.get('ci_95_low',0)*100:.1f}-{res.get('ci_95_high',0)*100:.1f}]")
    else:
        for sub, sub_res in res.items():
            if isinstance(sub_res, dict) and 'accuracy' in sub_res:
                print(f"{bench}.{sub:15s}: {sub_res['accuracy']*100:5.1f}%  n={sub_res['n']}")

In [ ]:
import json, subprocess
from pathlib import Path
pub_dir = Path('/kaggle/working/bench-out')
metadata = {'title': 'Caracal Bench s07 v0',
            'id': OUTPUT_DATASET,
            'licenses': [{'name': 'Apache-2.0'}]}
(pub_dir / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2))
r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(pub_dir), '--public'],
                   capture_output=True, text=True, check=False)
print(r.stdout, r.stderr)
if r.returncode != 0:
    subprocess.run(['kaggle', 'datasets', 'version', '-p', str(pub_dir),
                    '-m', 'bench s07 v0'], check=True)
print(f'Published -> {OUTPUT_DATASET}')